In [1]:
# Social Media Analytics Project (used for Academic Full Search)
# To extract tweets (with media info and not retweets) from Twitter using the tweepy Python library, tweepy (4.3.0)
# Author: Tena Bao

# Instructions for execution
# Prepare authentication token
# Input start_time and end_time before script execution

# Version 1.0, Feb 08, 2022:
# Use key words to earch for tweets posted within a time range.
# Create csv file "TwitterFullImagesData_YYYYMMDD.csv" and folder "Twitter Full Images Download"
# Version 1.1, Feb 09, 2022. Add logging to "get_full_photos_tweepy_log_YYYY-MM-DD_HH-MM.log"
# Version 1.2, Feb 10, 2022. Add total % 15000 == 0, sleep_time = 600
# Version 2.2 (TBT). Use authentication method "OAuth 2.0 App-only" and update "keys.py", update error handling with total % 15000 == 0

import tweepy
import keys
import datetime
import pandas as pd

import urllib.request
import os
import os.path

# For image download and csv file generation
import requests
import numpy as np
import time
from datetime import datetime, timedelta

import logging


In [2]:
# Logging File
log_file_name = datetime.now().strftime('get_full_photos_tweepy_log_%Y-%m-%d_%H-%M')
logging.basicConfig(filename=log_file_name, format='%(asctime)s|%(levelname)s|%(message)s', level=logging.DEBUG)#encoding='utf-8'

# To check if folder or file exists
def check_path_exist(file_name):
    file_exist = os.path.exists(file_name)
    if (file_exist):            
        return True
    else:
        return False    
      
# To create a folder to save images
# C:/Users/Tena/socialMediaEnv/Twitter Images Download_YYYYMMDD
# YYYYMMDD is today's date
date_of_today = datetime.today().strftime('%Y%m%d')
parent_dir = 'C:/Users/Tena/socialMediaEnv/'
directory = 'Twitter Full Images Download' + '_' + date_of_today
csv_file_name = 'TwitterFullImagesData' + '_' + date_of_today + '.csv'
 

def create_folder (parent_dir, dir):
    path = os.path.join(parent_dir, dir)
    if(check_path_exist(dir)):
        logging.error("Folder \"" + dir + "\" exists!")
        logging.shutdown()
        raise Exception("Folder \"" + dir + "\" exists!")  
        
    else:
        #try:
            os.mkdir(path)
            print("Folder '% s' created" % dir)
        #except OSError as error:
            #print(error)          
 
def check_csv (file_name):
    if (check_path_exist(file_name)):
        logging.error("Csv file \"" + file_name + "\" exists!")
        logging.shutdown()
        raise Exception("Csv file \"" + file_name + "\" exists!")         

# To download images to folders
def download_file (file_url, dir):
    if(file_url == '' and len(file_url) == 0):
        #print('Url is blank!')
        logging.error('Url is blank!')
    else:
        download_folder = parent_dir + dir + '/'
        file_name = file_url.split('/')[-1].split('.')[0]
        file_ext = '.'+file_url.split('.')[-1]
        image_file = requests.get(file_url)
        open(download_folder + file_name+file_ext, 'wb').write(image_file.content)
        
        #print('skip download')        
        
create_folder (parent_dir, directory)
check_csv(csv_file_name)
    
# Authenticating with Twitter Via Tweepy
# Creating and Configuring an OAuthHandler to Authenticate with Twitter
try:
    consumer_key = keys.consumer_key
    consumer_secret = keys.consumer_secret
    access_token = keys.access_token
    access_token_secret = keys.access_token_secret
except KeyError:
    sys.exit(1)
        
auth = tweepy.OAuthHandler(consumer_key, consumer_secret)
auth.set_access_token(access_token, access_token_secret)
api = tweepy.API(auth, wait_on_rate_limit=True, )

client = tweepy.Client(bearer_token = keys.bearer_token)


Folder 'Twitter Full Images Download_20220210' created


In [3]:
# This endpoint can deliver up to 100 Tweets per request in reverse-chronological order
# Fetch tweet_fields: 'author_id', 'conversation_id', 'created_at',  'id', 'public_metrics' and 'referenced_tweets'
# media_fields: 'media_key', 'type', 'preview_image_url', 'alt_text' and 'url'
# Coordinated Universal Time is 5 hours ahead of US Eastern Time.
# -5:00 is the local time offset for Eastern Time (US & Canada)s.

query = '(((vaccine OR pfizer OR moderna OR "Johnson and Johnson" OR "Johnson & Johnson" OR J&J) booster) OR #BoosterVaccine OR #VaccineBooster OR #PfizerBooster OR #ModernaBooster OR #JohnsonAndJohnsonBooster OR #Johnson&JohnsonBooster OR #J&JBooster) -is:retweet has:images'

# To collect data for US Eastern Time period: '2021-12-17T00:00:01Z' - '2021-12-24T00:00:01Z'
# Coordinated Universal Time period: '2021-12-18T05:00:01Z' - '2021-12-24T05:00:01Z'
start_time = '2022-02-07T05:00:01Z'
end_time = '2022-02-08T05:00:01Z'

def get_start_end_time():
    date_format = '%Y-%m-%d'
    sub_time = 'T05:00:01Z'
    now = datetime.now()
    current_day = now.strftime(date_format)
    start_day = (now - timedelta(6)).strftime(date_format)

    start_time = start_day + sub_time
    end_time = current_day + sub_time    
    start_end_time = [start_time, end_time]
    
    return start_end_time

#start_end_time = get_start_end_time()
#start_time = start_end_time[0]
#end_time = start_end_time[1]

print('Extract Twitter images data from ' + start_time + ' to ' + end_time + '.')
logging.info('Extract Twitter images data from %s to %s.', start_time, end_time)

max_fetch_number = 100
flatten_limit = 100
sleep_time = 600

# Define data fields
tweet_f = ['author_id','conversation_id','created_at','id','in_reply_to_user_id','public_metrics','referenced_tweets']
user_f = ['id','created_at','username','location','verified']
place_f = ['id','full_name']
media_f = ['media_key','type','preview_image_url','alt_text','url']

def get_tweets_data(first_query, nt):
# Client.search_all_tweets(query, *, user_auth=False, end_time, expansions, max_results, media_fields, next_token, place_fields, poll_fields, since_id, start_time, tweet_fields, until_id, user_fields)
# Query statement can be 1024 characters long for Academic Research access.
# The Tweets returned by this endpoint count towards the Project-level Tweet cap: 10 million Tweets per month

    if(first_query == False):
        tweets_data = client.search_all_tweets(query = query,

                                         tweet_fields = tweet_f,
                                         place_fields = place_f,
                                         #IndentationError: expected an indented block
                                         #place_fields=['place_type', 'geo'], expansions='geo.place_id',
                                         user_fields = user_f, #expansions='author_id',

                                         media_fields = media_f, expansions='attachments.media_keys',
                                         #Use next_token
                                         next_token = [nt],
                                         start_time = start_time,
                                         end_time = end_time, 
                                         max_results = max_fetch_number)

    else:
          tweets_data = client.search_all_tweets(query = query,

                                         tweet_fields = tweet_f,
                                         place_fields = place_f,
                                         user_fields = user_f, 

                                         media_fields = media_f, expansions='attachments.media_keys',

                                         start_time = start_time,
                                         end_time = end_time, 
                                         max_results = max_fetch_number)
            
    # When a response is returned without a next_token value, it can be assumed that all results have been paged through.
    next_token = tweets_data.meta.get('next_token', '')#next_token = tweets_data.meta['next_token']

    #print('tweets_data: ', tweets_data.meta)
    #print('next_token is: ', next_token)   
           
    return tweets_data
    #'3_1472295758984589314': <Media media_key=3_1472295758984589314 type=photo>,...



Extract Twitter images data from 2022-02-07T05:00:01Z to 2022-02-08T05:00:01Z.


In [4]:
# Get tweet by tweet id
# https://developer.twitter.com/en/docs/twitter-api/tweets/lookup/api-reference/get-tweets-id
def get_one_media(id):
    get_tweet = client.get_tweet(id, tweet_fields = tweet_f,
                                 place_fields = place_f,
                                 user_fields  = user_f, 
                                 media_fields = media_f, 
                                 expansions   = ['attachments.media_keys'])
    
    return get_tweet
#print(get_one_media('1483983447274233858'))

In [5]:
# Returns count of Tweets that match a search query.
# Client.get_all_tweets_count(query, *, end_time, granularity, since_id, start_time, until_id)
total_counts_by_day = client.get_all_tweets_count(query = query, granularity = 'day')#fullSearch
#print('total_counts_by_day: ', total_counts_by_day)

Forbidden: 403 Forbidden

In [ ]:
# Returns count of Tweets for a specific period that match a search query.
total_counts_by_test_period = client.get_all_tweets_count(query = query, granularity = 'day',#fullSearch
                                                             start_time = start_time,
                                                             end_time = end_time)
print('total_counts_by_test_period: ', total_counts_by_test_period)

In [20]:
output = []

# Twitter allows 10 million Tweets per month for full search, set the cap to 10,000,000 to avoid dead loop
# Get a notification when cap_limit has been reached
cap_limit = 10000000

# The total number of tweets retrieved
counter_total = 0 
 
# New in Tweepy version 4.0. Paginator.flatten() flattens the data and iterates over each object.
# Set 'pagination_token' to the value of next_token for the next page of results
def use_paginator(first_query, pagination_token, media, counter_total):
    counter = 0
    if(first_query == False):
        paginator_data = tweepy.Paginator( client.search_all_tweets, query,
                               tweet_fields = tweet_f,
                               place_fields = place_f,
                               #geo data not available for recent search
                               #IndentationError: expected an indented block
                               #place_fields = ['place_type', 'geo'], expansions='geo.place_id',
                               user_fields = user_f,
                               media_fields = media_f, 
                               expansions= ['author_id', 'attachments.media_keys'],

                               #Use pagination_token
                               pagination_token = [pagination_token],  
                                          
                               start_time = start_time,
                               end_time = end_time,
                               max_results = max_fetch_number).flatten(flatten_limit)#max_result = 10 to 100
    else:
        paginator_data = tweepy.Paginator( client.search_all_tweets, query,                                          
                               tweet_fields = tweet_f,
                               place_fields = place_f,                               
                               user_fields = user_f,
                               media_fields = media_f, 
                               expansions= ['author_id', 'attachments.media_keys'],

                               start_time = start_time,
                               end_time = end_time,
                               max_results = max_fetch_number).flatten(limit = flatten_limit)#max_result = 10 to 100
        
    for tweet in paginator_data:
        author_id = tweet.author_id
        conversation_id = tweet.conversation_id
        created_at = tweet.created_at
        twt_id = tweet.id
        #print('twt_id:')
        #print(twt_id)
        in_reply_to_user_id = tweet.in_reply_to_user_id

        #{'retweet_count': 0, 'reply_count': 0, 'like_count': 0, 'quote_count': 0}
        tweet_public_metrics = tweet.public_metrics
        
        retweet_count = tweet_public_metrics['retweet_count']#retweet_count = tweet_public_metrics.get('retweet_count'), int
        reply_count = tweet_public_metrics['reply_count']
        like_count = tweet_public_metrics['like_count']
        quote_count = tweet_public_metrics['quote_count']

        text = tweet.text      
        
        # referenced_tweets.type, id(, id.author_id, conversation_id, created_at, public_metrics)
        ref_twt_type = 'None'
        ref_twt_id = 'None'
        
        referenced_tweets = tweet.referenced_tweets
        if referenced_tweets:
            #print('referenced_tweets: ', referenced_tweets)                
            ref_twt_type = referenced_tweets[0].type
            ref_twt_id = referenced_tweets[0].id
            #print('ref_twt_id:', ref_twt_id)
        #else:
            #print('No referenced tweets.')  
                
        # Additional query is needed to get user data, but using it will exceed rate limets for recent search.
        # Rate limets for recent search: 180 per User, 450 per App (requests per 15-minute window unless otherwise stated)
        # Error code: 429, https://developer.twitter.com/en/support/twitter-api/error-troubleshooting
        #get_user = client.get_user(id = author_id, 
                                   #user_fields = ['created_at', 'username', 'location', 'verified'],
                                   #expansions = ['pinned_tweet_id']) 
                                   #)                         
        #username =  get_user.data.username      
        
        # Fix tweet with svg file issue
        #attachments = tweet.data['attachments']
        try:
            attachments = tweet.data['attachments']
        except KeyError:
            #print('Attachments not found!') 
            logging.error('Attachments not found!')
            continue
            
        media_keys = attachments['media_keys']

        attch_media_key = ''
        attch_type = ''
        attch_url = ''
        attch_preview_image_url = ''
        attch_alt_text = 'no alt text'
        
        media_keys_length = len(media_keys)
        #print('media_key size: ', media_keys_length)

        #for i in range(len(media_keys)):
        #if (media_keys[i]).media_key:
            #attch_media_key = media[media_keys[i]].media_key
            #print attch_media_key

        # Twitter allowes users to post up to 4 images in a tweet.
        # To download all the images of a tweet and store the urls in a csv file like below:
        # https://pbs.twimg.com/media/FFPZi7-WQAQsMKH.jpg
        # https://pbs.twimg.com/media/FFUQShRXwAITGCa.jpg, https://pbs.twimg.com/media/FFUQTMYWYAcD-BW.jpg
                          
        # Fix tweepy result_count issue starts                
        #if media[media_keys[0]].media_key:
            #attch_media_key = media[media_keys[0]].media_key#attch_media_key = media[media_keys[0]].get('media_key')
        
        try:
            attch_media_key = media[media_keys[0]].media_key
        except KeyError:
            attch_media_key = get_one_media(twt_id).includes.get('media')[0]
            logging.error('attch_media_key KeyError!')
            continue
         # Fix tweepy result_count issue ends  

        attch_type = media[media_keys[0]].type 
        if attch_type == 'photo' and media[media_keys[0]].url:           
            attch_url = media[media_keys[0]].url #photo 
            download_file(attch_url, directory)

            # Fix tweet with one image and one video issue
            if media_keys_length == 2:
                url1 = media[media_keys[1]].get('url', '')
                attch_url = attch_url + ', ' + url1
                download_file(url1, directory)
                   
            if media_keys_length == 3:
                url1 = media[media_keys[1]].get('url', '')
                url2 = media[media_keys[2]].get('url', '')
                attch_url = attch_url + ', ' + url1 + ', ' + url2
                download_file(url1, directory)
                download_file(url2, directory)         

            if media_keys_length == 4:
                url1 = media[media_keys[1]].get('url', '')
                url2 = media[media_keys[2]].get('url', '')
                url3 = media[media_keys[3]].get('url', '')
                attch_url = attch_url + ', ' + url1 + ', ' + url2 + ', ' + url3
                download_file(url1, directory)
                download_file(url2, directory) 
                download_file(url3, directory)
            
            #type = type(media[media_keys[0]].preview_image_url) #'NoneType'            
      
        # Twitter allowes users to post only 1 gif or video file in a tweet.
        if attch_type == 'animated_gif' and media[media_keys[0]].preview_image_url:#animated_gif
            attch_preview_image_url = media[media_keys[0]].preview_image_url     
            download_file(attch_preview_image_url, directory)

        if media[media_keys[0]].alt_text:
            attch_alt_text = media[media_keys[0]].alt_text#attch_alt_text = media[media_keys[0]].get('alt_text', '') 
        
        ai_prefix = 'ai_'
        ci_prefix = 'ci_'
        ti_prefix = 'ti_'
        
        to_ui_prefix = 'to_ui_'
        if(in_reply_to_user_id == None):
            to_ui_prefix = ''
        
        ref_ti_prefix = 'rf_ti_'
        if(ref_twt_id == 'None'):
            ref_ti_prefix = ''
        
        record = {'author_id':ai_prefix+str(author_id), 'conversation_id':ci_prefix+str(conversation_id), 'twt_created_at':created_at, 'twt_id':ti_prefix+str(twt_id), 'in_reply_to_user_id':to_ui_prefix+str(in_reply_to_user_id),
                  'retweet_count':retweet_count, 'reply_count':reply_count, 'like_count':like_count, 'quote_count':quote_count, 'text':text,
                  'ref_twt_type':ref_twt_type, 'ref_twt_id':ref_ti_prefix+str(ref_twt_id),
                  'Media Key':attch_media_key, 'Media Type':attch_type, 'Image URL':attch_url, 'Preview Image URL':attch_preview_image_url, 'Media Alt Text':attch_alt_text}
                  
        #print(record)
        output.append(record)
        counter += 1
        counter_total +=1
        if(counter_total > cap_limit):  
            logging.error('counter_total is greater than cap_limit')
            raise Exception('counter_total is greater than cap_limit ' + str(cap_limit) + '!')
        
        #print('counter: ', counter)
        #print('counter_total: ', counter_total)
        #logging.info('counter_total: %s', counter_total)
    return counter_total

def search_by_page(first_query, next_token, flag, total):
    # Get some latest records for testing purpose
    #while (flag and total < 100):
    while flag:
        print('print next_token at loop start: ', next_token)
        
        if((total > 0) and (total % 15000 == 0)):
        #if((total >= 15000) and first_sleep == False):
            print('total before sleep: ', total)
            print('Go sleeping at: ', datetime.now()) 
            logging.info('Go sleeping.')
            time.sleep(sleep_time)
            
        tweets_data = get_tweets_data(first_query, next_token)
        media = {m["media_key"]: m for m in tweets_data.includes['media']}
        #print('size of media in loop: ', len(media))
        #print('media: ', media)
        
        if first_query == False:
            pagination_token = next_token
        else:
            pagination_token = ''
            
        total = use_paginator(first_query, pagination_token, media, total)
        next_token = tweets_data.meta.get('next_token', '')
        result_count = tweets_data.meta.get('result_count', 0)
        print('result_count in loop: ', result_count)
        
        if(next_token != '' and len(next_token)>0 and result_count > 0):
            flag = True
            print('flage is True!')
        else:
            flag = False
            print('flag is False!')
            
        # Set first_query to False after the initial enquery
        first_query = False
        print('print next_token at loop end: ', next_token)
        logging.info('next_token at loop end is: %s', next_token)
        
    else:
        print('Loop ends!') 
        logging.info('Loop ends!')
        
first_query = True
flag = True
next_token = ''

print('Starting Twitter images data extraction! The start date/time is: ', datetime.now())
logging.info('Starting Twitter images data extraction!')
search_by_page (first_query, next_token, flag, counter_total)    

# Save csv file    
df = pd.DataFrame(output)

# Set starting index
starting_index = 1
df.index = np.arange(starting_index, len(df)+starting_index)

# Fix characters encoding issue.
df.to_csv(csv_file_name, mode='a', header=True, encoding = 'utf_8_sig')  
print('"' + csv_file_name + '"' + ' generated successfully! The generation date/time is: ', datetime.now())
logging.info('"' + csv_file_name + '"' + ' generated successfully!')
logging.shutdown()


Starting Twitter images data extraction! The start date/time is:  2022-02-10 19:22:13.670980
print next_token at loop start:  


Forbidden: 403 Forbidden